# VirulentHunter benchmark

In [1]:
# library
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from Bio import SeqIO
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

import src

tqdm.pandas()

# directory
root = Path.cwd()
base_dir = os.path.join(root, "data", "VirulentHunter")
os.chdir(base_dir)

/home/djmoon/miniforge3/envs/VFTextSeq/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def preprocess(fasta_path, label_path, desc_path, tax_path, id_field="id", replace=True):
    df = src.fasta_to_df(fasta_path)
    label = pd.read_csv(label_path).drop(columns=["Unnamed: 0"]) # drop index
    df = df.merge(label)

    # interproscan
    scan_df = src.parse_interproscan_file(desc_path)
    desc_cols = ["Signature Description", "InterPro Description"]
    description_dict = src.build_interpro_description(scan_df, "id", desc_cols)
    df["desc"] = df["id"].map(lambda x: description_dict.get(str(x), ""))
    model = SentenceTransformer("all-MiniLM-L6-v2")
    df["desc_nodup"] = df["desc"].progress_apply(
        lambda x: src.remove_semantic_duplicates_from_pipe_separated(x, model, 0.85)
    )

    # taxonomy
    tax_df = src.parse_mmseqs_lca(tax_path)
    tax_cols = df["id"].apply(lambda x: map_by_substring(x, tax_df))
    df["taxid"] = tax_cols.apply(lambda x: x[0])
    df["rank"] = tax_cols.apply(lambda x: x[1])
    df["name"] = tax_cols.apply(lambda x: x[2])

    if replace:
        mask = df["sequence"].str.contains("J|\\*", regex=True, na=False)
        print("rows with J or *:", mask.sum())
        df.loc[mask, "sequence"] = df.loc[mask, "sequence"].str.replace("J|\\*", "X", regex=True)
    
    df = df.drop_duplicates(subset=["id"], keep="first").reset_index(drop=True)

    return df

train = preprocess("train.fasta", "train_labels.csv", "virulent_output.tsv", "alnRes_lca_gtdb.tsv")
val   = preprocess("val.fasta",   "val_labels.csv", "virulent_output.tsv", "alnRes_lca_gtdb.tsv")
test  = preprocess("test.fasta",  "test_labels.csv", "virulent_output.tsv", "alnRes_lca_gtdb.tsv")

In [2]:
def preprocess(fasta_path, label_path, desc_path, tax_path, id_field="id", replace=True):
    df = src.fasta_to_df(fasta_path)
    label = pd.read_csv(label_path)
    df = df.merge(label)

    # interproscan
    scan_df = pd.read_csv(desc_path)
    df = df.merge(scan_df)

    # taxonomy
    tax_df = pd.read_csv(tax_path)
    df = df.merge(tax_df)

    if replace:
        mask = df["sequence"].str.contains("J|\\*", regex=True, na=False)
        print("rows with J or *:", mask.sum())
        df.loc[mask, "sequence"] = df.loc[mask, "sequence"].str.replace("J|\\*", "X", regex=True)
    
    df = df.drop(columns=["Unnamed: 0", "taxid"]) # drop index
    df = df.drop_duplicates(subset=["id"], keep="first").reset_index(drop=True)

    return df

train = preprocess("train.fasta", "train_labels.csv", "df_interproscan_no_dup_semantic.csv", "df_taxonomy_gtdb.csv")
val   = preprocess("val.fasta",   "val_labels.csv", "df_interproscan_no_dup_semantic.csv", "df_taxonomy_gtdb.csv")
test  = preprocess("test.fasta",  "test_labels.csv", "df_interproscan_no_dup_semantic.csv", "df_taxonomy_gtdb.csv")

rows with J or *: 4
rows with J or *: 0
rows with J or *: 0


In [3]:
X_train, y_train = src.esm_extract(train)
X_val,   y_val   = src.esm_extract(val)
X_test,  y_test  = src.esm_extract(test)

Using cache found in /home/djmoon/.cache/torch/hub/facebookresearch_esm_main
100%|██████████| 23067/23067 [22:59<00:00, 16.72it/s]


23067


Using cache found in /home/djmoon/.cache/torch/hub/facebookresearch_esm_main
100%|██████████| 2883/2883 [02:46<00:00, 17.36it/s]
Using cache found in /home/djmoon/.cache/torch/hub/facebookresearch_esm_main


2883


100%|██████████| 2884/2884 [02:49<00:00, 17.01it/s]


2884


In [3]:
from transformers import AutoTokenizer, AutoModel

def bert_extract(df):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
    model = AutoModel.from_pretrained("google-bert/bert-base-uncased").to(device)
    model.eval()
    txt_emb = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Extracting"):
        desc = str(row.get('desc_nodup', '')).strip()
        # name = str(row.get('name', '')).strip()
        # rank = str(row.get('rank', '')).strip()
        # combined_text = f"{desc}|{name}|{rank}"
        embedding = src.get_embedding(desc, tokenizer, model, device)
        txt_emb.append(embedding.numpy().tolist())
    
    return np.array(txt_emb)

In [4]:
X_train_text = bert_extract(train)
X_val_text   = bert_extract(val)
X_test_text  = bert_extract(test)

Extracting: 100%|██████████| 2884/2884 [00:15<00:00, 185.73it/s]


In [ ]:
def load_multiple_embeddings(df, emb_dirs):
    """
    Load and concatenate embeddings from multiple directories for the same ids in df.

    Args:
        df: DataFrame with columns ['id', 'label']
        emb_dirs: list of 3 paths to embedding directories

    Returns:
        features: np.ndarray of concatenated embeddings from all three sources
        labels: np.ndarray of labels
    """
    features = []
    labels = []

    for idx in tqdm(range(len(df)), desc="Loading embeddings"):
        id_ = df.iloc[idx]['id']
        label = df.iloc[idx]['label']

        emb_list = []
        for emb_dir in emb_dirs:
            emb_path = os.path.join(emb_dir, f'{id_}.pt')
            emb = torch.load(emb_path)
            emb_np = emb.numpy()

            if emb_np.ndim == 2:
                emb_np = emb_np.mean(axis=0)
            elif emb_np.ndim != 1:
                raise ValueError(f'Unexpected embedding shape: {emb_np.shape} for id {id_} in {emb_dir}')

            emb_list.append(emb_np)

        concatenated_emb = np.concatenate(emb_list)
        features.append(concatenated_emb)
        labels.append(label)

    features = np.stack(features)
    labels = np.array(labels)
    return features, labels

# reduced duplicates

In [17]:
X_train = np.load("embedding/esm_train.npy")
# X_train_text = np.load("embedding/bert_train.npy")
X_train_clip = np.load("embedding/clip_train.npy")
X_train_prot = np.load("embedding/prot_train.npy")

X_val = np.load("embedding/esm_val.npy")
# X_val_text = np.load("embedding/bert_val.npy")
X_val_clip = np.load("embedding/clip_val.npy")
X_val_prot = np.load("embedding/prot_val.npy")

X_test = np.load("embedding/esm_test.npy")
# X_test_text = np.load("embedding/bert_test.npy")
X_test_clip = np.load("embedding/clip_test.npy")
X_test_prot = np.load("embedding/prot_test.npy")

X_train_all = np.concatenate([X_train, X_train_text, X_train_prot, X_train_clip], axis=1)
X_val_all   = np.concatenate([X_val, X_val_text, X_val_prot, X_val_clip], axis=1)
X_test_all  = np.concatenate([X_test, X_test_text, X_test_prot, X_test_clip], axis=1)

In [18]:
y_train = train.label.values
y_val = val.label.values
y_test = test.label.values

print(y_train.shape)

if y_train.ndim != 1:
    y_train = np.asarray(y_train).reshape(-1).astype(int)
    y_val   = np.asarray(y_val).reshape(-1).astype(int)
    y_test  = np.asarray(y_test).reshape(-1).astype(int)

_X_train = np.concatenate([X_train_all, X_val_all], axis=0)
_y_train = np.concatenate([y_train, y_val], axis=0)

(23067,)


In [45]:
import warnings
warnings.filterwarnings(action='ignore', category=FutureWarning, module='xgboost')

from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

# Define the parameter grid to search
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.7, 1.0],
    'colsample_bytree': [0.7, 1.0]
}

# Initialize XGBClassifier
xgb = XGBClassifier(
    tree_method="hist",
    eval_metric="auc",
    # device="cuda:5",
    random_state=42,
)

# Grid search with 5-fold cross-validation
grid_search = GridSearchCV(estimator=xgb, param_grid=param_grid, cv=5, scoring='roc_auc', verbose=1, n_jobs=-1, error_score="raise")

# Fit grid search
grid_search.fit(_X_train, _y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best cross-validation AUC: ", grid_search.best_score_)

# Evaluate on test data using best estimator
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_all)
y_proba = best_model.predict_proba(X_test_all)[:, 1]

Fitting 5 folds for each of 108 candidates, totalling 540 fits


/home/djmoon/miniforge3/envs/VFTextSeq/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best parameters found:  {'colsample_bytree': 0.7, 'learning_rate': 0.2, 'max_depth': 7, 'n_estimators': 200, 'subsample': 1.0}
Best cross-validation AUC:  0.943668920148054


Best parameters found:  {'colsample_bytree': 0.7, 'learning_rate': 0.2, 'max_depth': 7, 'n_estimators': 200, 'subsample': 1.0}

In [19]:
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, matthews_corrcoef, confusion_matrix

best_params = {
    'colsample_bytree': 0.7,
    'learning_rate': 0.2,
    'max_depth': 7,
    'n_estimators': 200,
    'subsample': 1.0,
}

from xgboost import XGBClassifier

best_model = XGBClassifier(
    **best_params,
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
)

best_model.fit(_X_train, _y_train)
y_pred = best_model.predict(X_test_all)
y_proba = best_model.predict_proba(X_test_all)[:, 1]

print("Test ROC AUC:", roc_auc_score(y_test, y_proba))
print("Classification Report:\n", classification_report(y_test, y_pred))

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print(f"Accuracy: {accuracy:.3f}")
print(f"F1-score: {f1:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"MCC: {mcc:.3f}")
tn, fp, fn, tp = (confusion_matrix(y_test, y_pred)).ravel().tolist()
print("specificity", tn/(tn+fp))

Test ROC AUC: 0.9478439080411126
Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.90      0.89      1442
           1       0.90      0.86      0.88      1442

    accuracy                           0.88      2884
   macro avg       0.88      0.88      0.88      2884
weighted avg       0.88      0.88      0.88      2884

Accuracy: 0.883
F1-score: 0.881
Precision: 0.899
Recall: 0.864
MCC: 0.768
specificity 0.9029126213592233


In [39]:
print("Test ROC AUC:", roc_auc_score(y_test, y_proba))
print("Classification Report:\n", classification_report(y_test, y_pred))

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print(f"Accuracy: {accuracy:.3f}")
print(f"F1-score: {f1:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"MCC: {mcc:.3f}")

Test ROC AUC: 0.9513618106305582
Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.90      0.89      1442
           1       0.90      0.87      0.89      1442

    accuracy                           0.89      2884
   macro avg       0.89      0.89      0.89      2884
weighted avg       0.89      0.89      0.89      2884

Accuracy: 0.887
F1-score: 0.885
Precision: 0.897
Recall: 0.873
MCC: 0.774


In [29]:
print("Test ROC AUC:", roc_auc_score(y_test, y_proba))
print("Classification Report:\n", classification_report(y_test, y_pred))

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print(f"Accuracy: {accuracy:.3f}")
print(f"F1-score: {f1:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"MCC: {mcc:.3f}")

Test ROC AUC: 0.9431626689699351
Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.89      0.88      1442
           1       0.88      0.86      0.87      1442

    accuracy                           0.88      2884
   macro avg       0.88      0.88      0.88      2884
weighted avg       0.88      0.88      0.88      2884

Accuracy: 0.876
F1-score: 0.874
Precision: 0.884
Recall: 0.865
MCC: 0.751


In [10]:
print("Test ROC AUC:", roc_auc_score(y_test, y_proba))
print("Classification Report:\n", classification_report(y_test, y_pred))

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print(f"Accuracy: {accuracy:.3f}")
print(f"F1-score: {f1:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"MCC: {mcc:.3f}")

Test ROC AUC: 0.9419555691067077
Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.89      0.88      1442
           1       0.88      0.87      0.87      1442

    accuracy                           0.88      2884
   macro avg       0.88      0.88      0.88      2884
weighted avg       0.88      0.88      0.88      2884

Accuracy: 0.876
F1-score: 0.875
Precision: 0.884
Recall: 0.866
MCC: 0.753


In [ ]:
import joblib

# After training and grid search as before
best_model = grid_search.best_estimator_

# Save the model to a file
joblib.dump(best_model, 'best_binary_xgb_model.pkl')

# Later, you can load it back with
# loaded_model = joblib.load('best_xgb_model.pkl')


In [ ]:
import pandas as pd
import joblib

# Load the trained model
loaded_model = joblib.load('best_binary_xgb_model.pkl')

# Load your test set (replace 'test.csv' with your actual test file path)
df_test = pd.read_csv('binary/test_labels.csv')

labels = df_test['label']

# Predict
probabilities = loaded_model.predict_proba(X_test)[:,1]
predictions = loaded_model.predict(X_test)


# Prepare result dataframe with id, predicted label, and original label
result_df = pd.DataFrame({
    'id': df_test['id'],
    'prob': probabilities,
    'pred': predictions,
    'label': labels
})

# Save results
result_df.to_csv('binary/VFTextSeq_prediction_best_xgb.csv', index=False)


In [ ]:
result_df